# 01 — Readmission Prediction

Predict 30-day hospital readmission using the RETAIN model.
This is the core PyHealth 4-step pattern — all other clinical task notebooks follow the same structure.

**Task**: Binary classification (readmitted within 30 days: yes/no)  
**Model**: RETAIN (Reverse Time Attention — interpretable via attention weights)  
**Metrics**: PR-AUC, ROC-AUC, F1

In [ ]:
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset

ds = SyntheticEHRDataset()
ds.load()
print(f"Patients: {ds.get_patient_count()}, Visits: {ds.get_visit_count()}")

In [ ]:
# Attach the readmission task
# This transforms raw EHR data into (patient_history, readmission_label) pairs
from pyhealth.tasks import readmission_prediction_mimic3_fn
from pyhealth.datasets import split_by_patient, get_dataloader

task_dataset = ds.dataset.set_task(readmission_prediction_mimic3_fn)
task_dataset.stat()

train, val, test = split_by_patient(task_dataset, [0.8, 0.1, 0.1])
train_loader = get_dataloader(train, batch_size=32, shuffle=True)
val_loader   = get_dataloader(val,   batch_size=32, shuffle=False)
test_loader  = get_dataloader(test,  batch_size=32, shuffle=False)

In [ ]:
# RETAIN: uses visit-level (alpha) and code-level (beta) attention
# making it interpretable — you can see which visits/codes drove the prediction
from pyhealth.models import RETAIN

model = RETAIN(
    dataset=task_dataset,
    feature_keys=["conditions", "drugs"],
    label_key="readmission",
    mode="binary",
    embedding_dim=128,
    dropout=0.5,
)

In [ ]:
from pyhealth.trainer import Trainer

trainer = Trainer(model=model, metrics=["pr_auc", "roc_auc", "f1"])
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,          # reduce to 5 for a quick test
    monitor="pr_auc",
)

In [ ]:
result = trainer.evaluate(test_loader)
import pandas as pd
pd.Series(result).to_frame("RETAIN").round(4)

In [ ]:
# Compare with Transformer baseline
from pyhealth.models import Transformer

transformer = Transformer(
    dataset=task_dataset,
    feature_keys=["conditions", "drugs"],
    label_key="readmission",
    mode="binary",
    embedding_dim=128,
    nhead=4,
    num_encoder_layers=2,
    dropout=0.1,
)
t_trainer = Trainer(model=transformer, metrics=["pr_auc", "roc_auc", "f1"])
t_trainer.train(train_dataloader=train_loader, val_dataloader=val_loader,
                epochs=50, monitor="pr_auc")
t_result = t_trainer.evaluate(test_loader)

comparison = pd.DataFrame({"RETAIN": result, "Transformer": t_result}).round(4)
print(comparison)

## Using the Enterprise Package

The same pipeline above is encapsulated in `pyhealth_enterprise`:

In [ ]:
from pyhealth_enterprise.tasks.readmission import setup_readmission_task
from pyhealth_enterprise.models.registry import ModelName, get_model
from pyhealth_enterprise.pipelines.batch_risk_scorer import BatchRiskScorer

train_l, val_l, test_l = setup_readmission_task(ds.dataset)
m = get_model(ModelName.RETAIN, task_dataset, ["conditions", "drugs"], "readmission")
scorer = BatchRiskScorer(m, model_name="retain_readmission")
scorer.train(train_l, val_l, epochs=5)
risk_df = scorer.score_batch(test_l)
risk_df.head(10)